# Training of our own detector model (issue #46, work package 7)

Runs `training/model/train.py` in a GPU session on Colab (or Kaggle). What the script does and how a run is continued: `training/model/README.md`.

Before the first run:
- **Dataset.** Copy the folder `training/data/dataset/v1` (`train.jsonl`, `validation.jsonl`, `test.jsonl`) and `training/data/eval_suite.jsonl` into one folder of your Google Drive. The dev half of the suite chooses the epoch. Do not share the folder: not every source may be redistributed. Delete it when the training is done.
- **GPU.** Runtime → Change runtime type → GPU. (Kaggle: Settings → Accelerator → GPU, Internet on, and the files as a **private** dataset.)

One run per session. The folder of the run is on the Drive, so a session that ends before the run is done is continued by running the notebook again unchanged.

In [ ]:
RUN = "ettin-68m"                    # a candidate of model/train.py --list (not the name of a variant: that is OUT_NAME)
COMMIT = "own-model-training-57"     # branch or commit of the repository to train with; a commit for a result that is reported
HOURS = 11                           # stop and save before the session ends
DATA = "/content/drive/MyDrive/v1"   # folder with train.jsonl, validation.jsonl, test.jsonl and eval_suite.jsonl (Kaggle: /kaggle/input/<dataset>)
RUNS = "/content/drive/MyDrive/aivsai-runs"   # the run is written to RUNS/OUT_NAME (Kaggle: /kaggle/working/runs)
OUT_NAME = RUN                       # folder of the run; another name for a variant, e.g. "ettin-68m-last" with EXTRA = "--loss last"
RESUME_FROM = ""                     # Kaggle only: folder with the checkpoint.pt of an earlier session of this run
EXTRA = ""                           # further options of train.py, e.g. "--loss last", or "--rows 2000 --epochs 1" to try it out (with another RUNS)

In [ ]:
import os, pathlib

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
OUT = pathlib.Path(RUNS) / OUT_NAME
for name in ("train", "validation", "test", "eval_suite"):
    assert (pathlib.Path(DATA) / f"{name}.jsonl").exists(), f"{DATA}/{name}.jsonl is missing: set DATA to the folder with the dataset"

%cd $WORK
!rm -rf ai-check-extension && git clone -q https://github.com/OJPOJ/ai-check-extension.git && git -C ai-check-extension checkout -q $COMMIT && git -C ai-check-extension log -1 --oneline
!pip install -q "transformers>=5.17"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
resume = f"--resume-from {RESUME_FROM}" if RESUME_FROM else ""
%cd $WORK/ai-check-extension/training
!python model/train.py $RUN --data $DATA --suite $DATA/eval_suite.jsonl --out $OUT --hours $HOURS $resume $EXTRA

The folder of the run holds everything. When the run is done, copy `run.json`, `log.jsonl`, `scores/` and `model.pt` from it to `training/data/runs/<OUT_NAME>/` of the repository (with Google Drive for Desktop the folder is already on the computer); `checkpoint.pt` is no longer needed. Kaggle: to continue, save the version, add its output as input of the next session and set `RESUME_FROM` to it.

In [ ]:
!ls -la $OUT $OUT/scores 2>/dev/null